In [ ]:
%pip install torch torchvision torchaudio

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
 
# Laden des Datensatzes
df = pd.read_csv('../Datasets/Iris.csv')

# 2. Features (X) und Target (y) trennen
X = df.drop('species', axis=1).values
y = df['species'].values
# LabelEncoder sortiert die gefundenen Text-Kategorien standardmäßig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)
 
# Aufteilen des Datensatzes in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
 
# Umwandeln der Daten in PyTorch-Tensoren
# Hinweis: from_numpy() teilt Speicher mit NumPy (keine Kopie) – effizienter als torch.tensor()
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()
 
# Definieren des neuronalen Netzes als Sequential-Modell
net = nn.Sequential(
    nn.Linear(4, 5),   # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),           # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)
 
# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01)
 
# Trainieren des neuronalen Netzes
net.train()  # Trainingsmodus aktivieren
for epoch in range(100):
    optimizer.zero_grad()
    outputs = net(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
 
    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    net.train()
 
    print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')
 
# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)
 
# Abspeichern des trainierten Netzes
torch.save(net.state_dict(), '../Models/iris_net_NextStep0.pth')
joblib.dump(le, '../Models/label_encoder_NextStep0.pkl')

Epoch   0 | Loss: 1.7048
Epoch   1 | Loss: 1.5535
Epoch   2 | Loss: 1.4200
Epoch   3 | Loss: 1.3017
Epoch   4 | Loss: 1.1990
Epoch   5 | Loss: 1.1114
Epoch   6 | Loss: 1.0378
Epoch   7 | Loss: 0.9760
Epoch   8 | Loss: 0.9248
Epoch   9 | Loss: 0.8826
Epoch  10 | Loss: 0.8471
Epoch  11 | Loss: 0.8174
Epoch  12 | Loss: 0.7940
Epoch  13 | Loss: 0.7741
Epoch  14 | Loss: 0.7562
Epoch  15 | Loss: 0.7393
Epoch  16 | Loss: 0.7234
Epoch  17 | Loss: 0.7082
Epoch  18 | Loss: 0.6937
Epoch  19 | Loss: 0.6798
Epoch  20 | Loss: 0.6664
Epoch  21 | Loss: 0.6534
Epoch  22 | Loss: 0.6407
Epoch  23 | Loss: 0.6279
Epoch  24 | Loss: 0.6152
Epoch  25 | Loss: 0.6024
Epoch  26 | Loss: 0.5896
Epoch  27 | Loss: 0.5770
Epoch  28 | Loss: 0.5648
Epoch  29 | Loss: 0.5532
Epoch  30 | Loss: 0.5424
Epoch  31 | Loss: 0.5324
Epoch  32 | Loss: 0.5232
Epoch  33 | Loss: 0.5147
Epoch  34 | Loss: 0.5067
Epoch  35 | Loss: 0.4990
Epoch  36 | Loss: 0.4914
Epoch  37 | Loss: 0.4840
Epoch  38 | Loss: 0.4766
Epoch  39 | Loss: 0.4692


['../Models/label_encoder_NextStep0.pkl']

In [ ]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(4, 5),    # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),          # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)

net.load_state_dict(torch.load('../Models/iris_net_NextStep0.pth', map_location=torch.device('cpu')))
le = joblib.load('../Models/label_encoder_NextStep0.pkl')

for k, v in net.named_parameters():
    print(k,v)
net.eval()

In [ ]:
# Vorhersage mit dem trainierten Netz
# es wird eine Userabfrage gemacht, um die Merkmale einzugeben
# und eine Vorhersage zu treffen.

# Eingabe der vier Merkmale
print("Geben Sie die vier Merkmale ein:")
sepal_length = float(input("Sepal-Länge (cm): "))
sepal_width = float(input("Sepal-Breite (cm): "))
petal_length = float(input("Petal-Länge (cm): "))
petal_width = float(input("Petal-Breite (cm): "))

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor([[sepal_length, sepal_width, petal_length, petal_width]], dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage
print("Klasse: ", le.inverse_transform([predicted.item()])[0])

In [ ]:
# aus dem obigen Skript wird inputs übernommen, 
# die Vorhersagen werden als Wahrscheinlichkeiten ausgegeben

import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", max_index.item())

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)
print("Klassenwahrscheinlichkeiten:", probabilities.numpy()[0])
